**SPARK SQL**

In [1]:
from spark_utils import SparkUtils

In [2]:
spark_url = "spark://spark-master:7077"
app_name = "Spark SQL - Example 1"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/19 00:33:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType
data = [
(1, "Alice", 29),
(2, "Bob", 35),
(3, "Charlie", 41)
]

schema = StructType([
    StructField("id",IntegerType(), True),
    StructField("name",StringType(), True),
    StructField("age",IntegerType(), True)
])

df = su._spark.createDataFrame(data, schema)
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)



In [4]:
from pyspark.sql.types import StructField, StructType, TimestampType, FloatType
from datetime import datetime

factory_data = [
    ("M001", datetime(2026, 1, 26, 8, 0, 0), 75.3),
    ("M002", datetime(2026, 1, 26, 8, 5, 0), 68.7),
    ("M001", datetime(2026, 1, 26, 8, 10, 0), 76.1),
    ("M003", datetime(2026, 1, 26, 8, 15, 0), 72.4),
    ("M002", datetime(2026, 1, 26, 8, 20, 0), 69.8),
    ("M001", datetime(2026, 1, 26, 8, 25, 0), 77.5),
    ("M003", datetime(2026, 1, 26, 8, 30, 0), 73.2),
    ("M002", datetime(2026, 1, 26, 8, 35, 0), 70.1),
    ("M001", datetime(2026, 1, 26, 8, 40, 0), 78.0),
    ("M003", datetime(2026, 1, 26, 8, 45, 0), 74.6),
]

factory_schema = StructType([
    StructField("id",StringType(), True),
    StructField("date",TimestampType(), True),
    StructField("temp",FloatType(), True)
])

In [5]:
factory_schema

StructType([StructField('id', StringType(), True), StructField('date', TimestampType(), True), StructField('temp', FloatType(), True)])

In [6]:
factory_df = su._spark.createDataFrame(factory_data, factory_schema)
factory_df.printSchema()


root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temp: float (nullable = true)



In [7]:
print(f"Number of records:{factory_df.count()}")

[Stage 0:======================================================>  (21 + 1) / 22]

Number of records:10


In [8]:
from pyspark.sql.functions import col
df_filtered = factory_df.filter(col("temp") > 70)
df_filtered.show(truncate = False)

+----+-------------------+----+
|id  |date               |temp|
+----+-------------------+----+
|M001|2026-01-26 08:00:00|75.3|
|M001|2026-01-26 08:10:00|76.1|
|M003|2026-01-26 08:15:00|72.4|
|M001|2026-01-26 08:25:00|77.5|
|M003|2026-01-26 08:30:00|73.2|
|M002|2026-01-26 08:35:00|70.1|
|M001|2026-01-26 08:40:00|78.0|
|M003|2026-01-26 08:45:00|74.6|
+----+-------------------+----+



In [9]:
factory_df = factory_df.orderBy(col("temp"),ascending=False)
factory_df.show()

[Stage 6:=================================>                       (13 + 9) / 22]

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:00:00|75.3|
|M003|2026-01-26 08:45:00|74.6|
|M003|2026-01-26 08:30:00|73.2|
|M003|2026-01-26 08:15:00|72.4|
|M002|2026-01-26 08:35:00|70.1|
|M002|2026-01-26 08:20:00|69.8|
|M002|2026-01-26 08:05:00|68.7|
+----+-------------------+----+



In [10]:
factory_df = factory_df.orderBy(col("temp"),ascending=False).limit(5)
factory_df.show()

[Stage 7:============================================>            (17 + 5) / 22]

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:00:00|75.3|
|M003|2026-01-26 08:45:00|74.6|
+----+-------------------+----+



In [11]:
factory_groupped = factory_df.groupBy(col("machine_id"))
factory_groupped

GroupedData[grouping expressions: [machine_id], value: [id: string, date: timestamp ... 1 more field], type: GroupBy]

In [12]:
factory_groupped = factory_df.groupBy(col("id")).count()
factory_groupped.show()

[Stage 8:=================================>                       (13 + 9) / 22]

+----+-----+
|  id|count|
+----+-----+
|M001|    4|
|M003|    1|
+----+-----+



In [15]:
from pyspark.sql.functions import avg,min, max
agg_df = factory_df.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp")
)
agg_df.show()

[Stage 12:===========================>                           (11 + 11) / 22]

+----+-----------------+--------+
|  id|         avg_temp|min_temp|
+----+-----------------+--------+
|M001|76.72500038146973|    75.3|
|M003| 74.5999984741211|    74.6|
+----+-----------------+--------+



***Smart Factory Sensor Data Analysis***

In [16]:
factory_df = su._spark.createDataFrame(factory_data,factory_schema)

Machine 1

In [17]:
M001 = factory_df.filter(col("id") ==  "M001")
M001.show()

agg_M001 = M001.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)

agg_M001.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:00:00|75.3|
|M001|2026-01-26 08:10:00|76.1|
|M001|2026-01-26 08:25:00|77.5|
|M001|2026-01-26 08:40:00|78.0|
+----+-------------------+----+



[Stage 17:=================>                                      (7 + 15) / 22]

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M001|76.72500038146973|    75.3|    78.0|
+----+-----------------+--------+--------+



Machine 2

In [18]:
M002 = factory_df.filter(col("id") ==  "M002")
M002.show()

agg_M002 = M002.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)

agg_M002.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M002|2026-01-26 08:05:00|68.7|
|M002|2026-01-26 08:20:00|69.8|
|M002|2026-01-26 08:35:00|70.1|
+----+-------------------+----+

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M002|69.53333282470703|    68.7|    70.1|
+----+-----------------+--------+--------+



Machine 3

In [19]:
M003 = factory_df.filter(col("id") ==  "M003")
M002.show()

agg_M003 = M003.groupBy(col("id")).agg(
    avg("temp").alias("avg_temp"),
    min("temp").alias("min_temp"),
    max("temp").alias("max_temp")
)

agg_M003.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M002|2026-01-26 08:05:00|68.7|
|M002|2026-01-26 08:20:00|69.8|
|M002|2026-01-26 08:35:00|70.1|
+----+-------------------+----+

+----+-----------------+--------+--------+
|  id|         avg_temp|min_temp|max_temp|
+----+-----------------+--------+--------+
|M003|73.39999898274739|    72.4|    74.6|
+----+-----------------+--------+--------+



Filter records above a temperature threshold (temp > 75)

In [20]:
df_filtered = factory_df.filter(col("temp") < 75)
df_filtered.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M002|2026-01-26 08:05:00|68.7|
|M003|2026-01-26 08:15:00|72.4|
|M002|2026-01-26 08:20:00|69.8|
|M003|2026-01-26 08:30:00|73.2|
|M002|2026-01-26 08:35:00|70.1|
|M003|2026-01-26 08:45:00|74.6|
+----+-------------------+----+



Count the number of readings per machine

In [21]:
print("M001 Readings: " + str(M001.count()))
print("M002 Readings: " + str(M002.count()))
print("M003 Readings: " + str(M003.count()))

M001 Readings: 4
M002 Readings: 3
M003 Readings: 3


Find the machine with the highest temperature

In [22]:
max_df = factory_df.orderBy(col("temp").desc()).limit(1)

max_df.show()

+----+-------------------+----+
|  id|               date|temp|
+----+-------------------+----+
|M001|2026-01-26 08:40:00|78.0|
+----+-------------------+----+



In [ ]:
su._spark.stop()